# 📊 Exploration autonome

> ✍️ **Notebook dirigé** · complétez les cellules marquées ✍️ À VOUS.

**Jour 1 · 9h15** · 30 minutes, sans consignes techniques

*Analyse exploratoire et interprétation métier*

**Objectifs**
- Faire un diagnostic rapide de votre jeu de données
- Identifier les variables clés pour le commanditaire
- Formuler trois premières hypothèses

## Le principe
Pendant 30 minutes, vous explorez **comme vous voulez**. Aucune consigne technique : seulement des questions auxquelles répondre à la fin.

1. Que représente une ligne ?
2. Quelle période, combien de clients, combien de lignes ?
3. Quelles variables comptent pour le commanditaire de votre brief ?
4. Quels problèmes de qualité voyez-vous ?
5. Quelles **trois hypothèses** voulez-vous tester aujourd'hui ?

In [1]:
# ⚙️ À exécuter en premier : récupère la configuration du cours (quelques secondes)
import os, sys
CONFIG_ID = "1bW5RsToUHc-7pGiyHS_1B2OYkAFsrv3r"

# Suppression du fichier corrompu s'il existe
if os.path.exists("config_projet.py"):
    os.remove("config_projet.py")

# Téléchargement propre avec gdown pour contourner l'avertissement de virus Google Drive
!gdown --id {CONFIG_ID} -O config_projet.py

print("✅ Configuration prête : chaque fichier de données se télécharge au premier usage")

✅ Configuration prête : chaque fichier de données se télécharge au premier usage


In [2]:
# Secteur traité : Crédit Azur (banque) — autres choix : "commerce", "hotellerie"
SECTEUR = "banque"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from config_projet import charger, reference, nettoyage_minimal

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
brut, cfg = charger(SECTEUR)
print(cfg["nom"], "·", brut.shape[0], "lignes,", brut.shape[1], "colonnes")

Crédit Azur · 20477 lignes, 20 colonnes


In [3]:
from config_projet import brief
brief(SECTEUR)

*(brief du secteur : fichier BRIEF_banque.md téléchargé depuis Drive dans Colab)*

In [4]:
# ✍️ Exploration libre : les six questions du diagnostic rapide
print("1. Une ligne = une demande ? id unique :", brut[cfg["id"]].is_unique,
      "· clients distincts :", brut["client_id"].nunique())
print("2. Période :", brut[cfg["date"]].min().date(), "→", brut[cfg["date"]].max().date())
mois = brut[cfg["date"]].dt.to_period("M")
print("   mois couverts :", mois.nunique(), "· mois manquants :",
      len(pd.period_range(mois.min(), mois.max(), freq="M").difference(mois.unique())))
print("3. Doublons exacts :", brut.duplicated().sum(),
      "· même demande_id répété :", brut[cfg["id"]].duplicated().sum())
print("4. Types :"); print(brut.dtypes.to_string())

print("\n5. Valeurs manquantes (%) :")
manq = brut.isna().mean().mul(100).round(1)
print(manq[manq > 0].to_string())
# Où manquent-elles ? Le « résultat » (remboursement) n'est connu que si le crédit a été accordé.
print("\n   Remboursement connu selon la décision :")
print(brut.groupby("credit_accorde")["rembourse_sans_incident"].apply(lambda s: s.notna().mean()).round(3).to_string())
print("   Revenu manquant selon le canal (%) :")
print(brut.groupby("canal")["revenu_mensuel"].apply(lambda s: s.isna().mean() * 100).round(1).to_string())

print("\n6. Valeurs impossibles : âge < 18 :", (brut["age"] < 18).sum(),
      "· âge max :", brut["age"].max(), "· PNB négatif :", (brut[cfg["ca"]] < 0).sum(),
      "· montant accordé > demandé :", (brut["montant_accorde"] > brut["montant_demande"]).sum())
display(brut.describe().T.round(2))

# Premières pistes de lecture : décision selon l'âge, et dans le temps
brut["periode"] = np.where(brut[cfg["date"]] < cfg["evenement"][0], "avant score V2", "après score V2")
print("\nTaux d'accord par tranche d'âge et période :")
display(brut.pivot_table(index="tranche_age", columns="periode", values="credit_accorde", aggfunc="mean").round(3))
brut = brut.drop(columns="periode")

1. Une ligne = une demande ? id unique : False · clients distincts : 11173
2. Période : 2023-01-01 → 2025-12-31
   mois couverts : 36 · mois manquants : 0
3. Doublons exacts : 82 · même demande_id répété : 82
4. Types :
date_demande               datetime64[us]
demande_id                            str
client_id                             str
genre                                 str
age                                 int64
tranche_age                           str
region                                str
canal                                 str
type_credit                           str
montant_demande                   float64
duree_mois                          int64
revenu_mensuel                    float64
taux_endettement                  float64
anciennete_emploi_ans             float64
contrat                               str
score_algo                        float64
credit_accorde                      int64
montant_accorde                   float64
pnb_estime_eur          

,count,mean,min,25%,50%,75%,max,std
date_demande,20477,2024-07-03 03:14:09.675245,2023-01-01 00:00:00,2023-10-04 00:00:00,2024-06-29 00:00:00,2025-04-07 00:00:00,2025-12-31 00:00:00,NaN
age,20477.0,42.320945,17.0,31.0,40.0,52.0,85.0,14.75177
montant_demande,20477.0,11130.990868,500.0,5600.0,9500.0,14800.0,86500.0,7454.138263
duree_mois,20477.0,46.156371,12.0,36.0,48.0,60.0,72.0,18.617209
revenu_mensuel,19895.0,2537.656698,530.0,1820.0,2370.0,3060.0,9710.0,984.872907
taux_endettement,20477.0,0.352357,0.05,0.22,0.317,0.447,0.9,0.177884
anciennete_emploi_ans,20477.0,7.517761,0.0,2.7,5.6,10.4,40.0,6.690554
score_algo,20477.0,0.686156,0.013,0.554,0.752,0.865,0.997,0.228422
credit_accorde,20477.0,0.639937,0.0,0.0,1.0,1.0,1.0,0.48003
montant_accorde,20477.0,6722.024711,0.0,0.0,4900.0,11000.0,86500.0,7449.156066



Taux d'accord par tranche d'âge et période :


periode,après score V2,avant score V2
tranche_age,,
18-24,0.077,0.361
25-34,0.343,0.604
35-44,0.616,0.815
45-54,0.779,0.911
55-64,0.831,0.939
65+,0.766,0.898


## Vos réponses
1. **Une ligne =** une *demande de crédit* (`demande_id`), et non un client : 20 477 lignes pour 11 173 clients, un même client pouvant redemander un crédit.
2. **Période et volumes :** du 01/01/2023 au 31/12/2025, 36 mois complets (aucun mois manquant), 20 477 lignes, 11 173 clients, 4 types de crédit (auto, travaux, personnel, renouvelable), 3 canaux (agence, web, courtier).
3. **Variables clés :** la décision `credit_accorde` (64 % d'accords) et `score_algo` (le score, corrélé à 0,83 avec la décision), le `pnb_estime_eur` (le chiffre d'affaires de la banque, nul pour un refus), puis les critères de risque (`taux_endettement`, `revenu_mensuel`, `contrat`, `anciennete_emploi_ans`) et les variables sensibles (`tranche_age`, `genre`, `region`).
4. **Problèmes de qualité :**
   - 82 doublons exacts (même `demande_id` saisi deux fois) ;
   - 11 demandes avec un âge de 17 ans (mineurs) ;
   - 2,8 % de `revenu_mensuel` manquants, répartis de façon homogène entre canaux (2,8 à 2,9 %), donc plutôt aléatoires ;
   - 36 % de `rembourse_sans_incident` manquants : ils ne manquent **pas au hasard**, ils correspondent exactement aux crédits refusés (résultat connu pour 100 % des accordés, 0 % des refusés) : c'est un biais de sélection ;
   - 36 % de PNB à zéro, correspondant aux refus : à ne pas lire comme des « petits clients ».
5. **Hypothèses :**
   - **H1** : le taux d'accord a chuté durablement à partir de mars 2024, au déploiement du score V2 (76 % d'accords avant, 57 % après d'après ce premier diagnostic), et c'est ce qui explique la baisse de production.
   - **H2** : les plus jeunes sont les plus touchés : à 18-24 ans, 36 % d'accords avant V2 contre 8 % après ; à 55-64 ans, 94 % contre 83 %.
   - **H3** : cet écart d'âge ne s'explique qu'en partie par des critères de risque légitimes (contrat, ancienneté) et ne peut pas être vérifié sur les refusés, dont on ne connaît jamais le remboursement.

*Note : le fichier `BRIEF_banque.md` n'est pas dans le dossier ; ces hypothèses s'appuient sur la présentation du cours (« Nouveau score de crédit en 2024 : la production baisse, les jeunes actifs se plaignent »).*

💾 **Versionnez** : *Fichier › Enregistrer une copie dans GitHub*, message de commit : « Exploration : hypothèses H1 à H3 »